# Short Story Studio V2 — Colab-first Creator Workspace

A notebook-native production dashboard for Short Story Speaker. It combines an LLM scene planner, editable scene board, free stock-footage search, voice preview, captions and an HTML/CSS/GSAP video renderer.

## What this version actually generates
- **Real footage:** searches Pexels or Pixabay using your own free API key, lets you select a clip per scene, then downloads the selected licensed stock footage.
- **Motion graphics:** creates animated text, titles, geometric backgrounds, subtitles and transitions using HyperFrames (Apache 2.0) + HTML/CSS/GSAP.
- **Voice:** Google Gemini 3.8 Flash TTS has a hosted free tier on eligible accounts and supports Hindi/Bangla; Chatterbox V3 is an optional open-source research TTS/voice-cloning model for Colab GPU.
- **Export:** MP4, SRT captions, timestamp JSON, storyboard JSON and a stock-source credit manifest.

**Important distinction:** Pexels/Pixabay provide real stock footage, not AI-generated video. The code-motion engine produces animated graphics, not photorealistic AI video. There is no reliable unlimited free AI text-to-video API; commercial video-model APIs normally use credits or billing. This notebook does not silently switch to a paid service.

The app runs inside notebook widgets. It does not expose a separate web server. When you stop the Colab runtime, the notebook session and in-memory API keys stop too. Free Colab GPU access is not guaranteed; the core stock-footage and motion-graphics renderer runs on CPU.


In [ ]:
from pathlib import Path
import subprocess, shutil, os, re

REPO_ROOT = Path('/content/VIDEO-GENERATIONS-TOOLS')
REPO_URL = 'https://github.com/SaidulM/VIDEO-GENERATIONS-TOOLS.git'
BRANCH = 'studio-v2'
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
%cd /content/VIDEO-GENERATIONS-TOOLS
%pip -q install -r requirements-colab-studio.txt
print('Short Story Studio V2 dependencies installed. No AI video checkpoints were downloaded.')


In [ ]:
# Install the open-source HTML-to-MP4 renderer. This downloads the CLI, not a video diffusion model.
import subprocess, shutil

def _run(command):
    print('>', ' '.join(command))
    subprocess.run(command, check=True)

node_path = shutil.which('node')
node_major = 0
if node_path:
    node_version = subprocess.check_output(['node', '--version'], text=True).strip()
    node_major = int(node_version.lstrip('v').split('.')[0])
if node_major < 22:
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -', shell=True, check=True)
    _run(['apt-get', 'install', '-y', 'nodejs'])
if not shutil.which('ffmpeg'):
    _run(['apt-get', 'update', '-qq'])
    _run(['apt-get', 'install', '-y', 'ffmpeg'])
_run(['npm', 'install', '--global', 'hyperframes@0.9.98'])
_run(['hyperframes', 'info'])
print('Renderer ready. Chromium is installed lazily on the first video render.')


## Optional: accurate audio-aligned captions

The studio can create sentence-level estimated captions without this package. Install Faster-Whisper only when you want word-timestamp alignment. It downloads the tiny speech-recognition model the first time you render captions.


In [ ]:
%pip -q install -r requirements-colab-whisper.txt
print('Faster-Whisper installed. The tiny transcription model downloads on first caption alignment.')


## Optional: Chatterbox V3 voice cloning

Chatterbox is an experimental open-source research TTS model with Hindi support. The upstream project is retired/no longer maintained. It requires a Colab CUDA GPU for a practical run on this setup. Install only when you want to test it; it may change the Colab PyTorch stack and may ask for a runtime restart. Use only your own voice sample or a voice you are authorized to use. Do not create deceptive impersonations.


In [ ]:
%pip -q install chatterbox-tts==0.1.7
print('Chatterbox package installed. Restart the runtime if Colab asks, then rerun Setup and the Dashboard cells.')


In [ ]:
# Optional Google Drive mount. Exports are saved there when available; otherwise they remain in this session.
from pathlib import Path
DRIVE_READY = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_READY = True
except Exception as exc:
    print('Google Drive not mounted; outputs will stay in the temporary Colab runtime.', exc)

if DRIVE_READY:
    EXPORT_ROOT = Path('/content/drive/MyDrive/ShortStoryStudioV2')
else:
    EXPORT_ROOT = Path('/content/short-story-studio-v2-exports')
WORK_ROOT = Path('/content/short-story-studio-v2-work')
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Exports:', EXPORT_ROOT)


## Open the creator dashboard

1. Configure an LLM provider and refresh its model list if you want topic-to-script generation. A pasted script can be split into scenes without an LLM API.
2. Edit scene narration, on-screen text and visual direction.
3. Add Pexels/Pixabay keys for real stock footage, or use code-generated motion graphics with no video key.
4. Preview your selected TTS voice before rendering.
5. Render and download the MP4, captions and timestamps.

API keys entered into the dashboard are kept in this notebook's live memory only; they are not written to GitHub or project files. A free-tier Gemini TTS request can be used by Google to improve products, so do not submit confidential text to that free tier.


In [ ]:
from studio_v2.ui import ColabStudio

STUDIO = ColabStudio(work_dir=WORK_ROOT, export_dir=EXPORT_ROOT)
STUDIO.display()
